# Experiment 4: DeepLOB under test-time perturbation

Evaluate the five experiment 3 models on their original October–December 2017 test days at **P = 1%, 5%, 10%, 25%, and 50%**. Reuse the saved P=0% scores as the baseline. Keep the weights, labels, 50-snapshot windows, k=20 horizon, and batch size 64 fixed; no retraining is needed.

Run all cells in the `deep-learning` environment. Sections 5–6 display the overall and per-class tables; Section 7 saves the numeric results. Full evaluation covers 87,931,530 perturbed test windows.


## 1. Set the stocks, probabilities, and device

Select all five stocks and the five P values below. Use CUDA when available, otherwise the Apple GPU or CPU, and process stocks sequentially. Set perturbation seed 42 for each stock and P so the probabilities share the same underlying random draws.


In [1]:
from pathlib import Path
import sys
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "experiment_04" and PROJECT_DIR.parent.name == "experiments":
    PROJECT_DIR = PROJECT_DIR.parent.parent
sys.path.insert(0, str(PROJECT_DIR))

from deeplob import DeepLOB
from train_deeplob import DailyWindowDataset, classification_metrics, run_epoch
from wse_lob import (
    OrderBook,
    compute_daily_statistics,
    compute_history_statistics,
    normalize_features,
)

torch.set_num_threads(1)
STOCKS = ("KGHM", "PEKAO", "PKNORLEN", "PKOBP", "PZU")
PERTURBATION_PROBABILITIES = (0.01, 0.05, 0.10, 0.25, 0.50)
PERTURBATION_SEED = 42
TICK_SIZE = 0.01
HISTORY_DAYS = 5
EXPERIMENT_DIR = PROJECT_DIR / "experiments" / "experiment_04"
BASELINE_DIR = PROJECT_DIR / "experiments" / "experiment_03"

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Device: {DEVICE}; stocks: {', '.join(STOCKS)}")


Device: mps; stocks: KGHM, PEKAO, PKNORLEN, PKOBP, PZU


## 2. Load the models and baseline scores

Read each checkpoint's configuration and test dates, then check that the daily files exist. Load experiment 3's saved P=0% metrics. Every P for a stock will use the same dates, labels, and eligible windows.


In [9]:
checkpoints = {}
baseline_metrics = {}
processed_dates = {}

for stock in STOCKS:
    checkpoint = torch.load(
        BASELINE_DIR / stock / "best_model.pt",
        map_location="cpu",
        weights_only=True,
    )
    config = checkpoint["config"]
    assert config["stock"] == stock
    assert (config["window_length"], config["n_levels"], config["horizon"], config["batch_size"]) == (50, 10, 20, 64)
    checkpoints[stock] = checkpoint

    with np.load(BASELINE_DIR / stock / "test_metrics.npz", allow_pickle=False) as arrays:
        baseline_metrics[stock] = dict(arrays)

    processed_dir = PROJECT_DIR / "data" / "wselob_2017_processed" / stock
    dates = sorted(path.stem for path in processed_dir.glob("*.npz"))
    processed_dates[stock] = dates
    for date in config["data_dates"]["test"]:
        index = dates.index(date)
        assert index >= HISTORY_DAYS
        for daily_path in (
            processed_dir / f"{date}.npz",
            PROJECT_DIR / config["feature_dir"] / f"{date}.npz",
            PROJECT_DIR / config["label_dir"] / f"{date}.npz",
        ):
            if not daily_path.is_file():
                raise FileNotFoundError(daily_path)

print("All five checkpoints and their 62 test days are available.")


All five checkpoints and their 62 test days are available.


## 3. Perturb raw features, then normalize

For each snapshot, `OrderBook.add_perturbation` processes ask then bid:

1. Find vacant ticks between each side's **original first and tenth levels**, using `tick_size=0.01`.
2. Independently fill each vacant tick with probability P and volume 1.
3. Sort and retain ten levels per side; keep the original array unchanged.

Normalize with the **original preceding five trading days**. Best quotes and mid-prices stay unchanged, so reuse the existing labels. Perturb each snapshot once before constructing overlapping windows.

This is our array-based implementation of the dataset example's probability rule. Vacant ticks are sampled in ascending price order; each P's generator advances across days. Larger P selects more of the original vacant ticks before top-ten truncation.


In [10]:
def prepare_perturbed_day(raw_features, probability, rng, mean, std):
    """Perturb one raw day and return float32 model features."""
    perturbed = OrderBook.add_perturbation(
        raw_features, probability=probability, rng=rng, tick_size=TICK_SIZE,
    )
    # Best ask/bid prices and their volumes must remain exactly unchanged.
    np.testing.assert_array_equal(perturbed[:, :4], raw_features[:, :4])
    return normalize_features(perturbed, mean, std).astype(np.float32)


## 4. Evaluate all stocks and probabilities

Restore each selected model, disable gradients and dropout, and evaluate one day at a time with the existing dataset and evaluation functions. For every day, verify that P=0% normalization exactly reproduces the saved prepared features.

Aggregate confusion counts and sample-weighted loss over all 62 test days, then calculate the scores. Check that every P has the baseline's class support. The next cell runs the full experiment and prints monthly progress.


In [11]:
def evaluate_stock(stock):
    """Evaluate all requested probabilities and return their full-test metrics."""
    checkpoint = checkpoints[stock]
    config = checkpoint["config"]
    model = DeepLOB(
        n_levels=config["n_levels"],
        batch_norm=config["batch_norm"],
        dropout=config["dropout"],
        initialization=config["initialization"],
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.requires_grad_(False)
    model.to(DEVICE)
    model.eval()
    if DEVICE.type == "cuda":
        torch.backends.cudnn.allow_tf32 = config["cudnn_tf32"]
        torch.backends.cuda.matmul.allow_tf32 = config["matmul_tf32"]
        torch.backends.cudnn.benchmark = config["cudnn_benchmark"]
    criterion = torch.nn.CrossEntropyLoss()

    processed_dir = PROJECT_DIR / "data" / "wselob_2017_processed" / stock
    feature_dir = PROJECT_DIR / config["feature_dir"]
    label_dir = PROJECT_DIR / config["label_dir"]
    test_dates = config["data_dates"]["test"]
    all_dates = processed_dates[stock]

    # Cache only small statistics; historical data always stays unperturbed.
    statistics = {}
    generators = {}
    confusion = {}
    loss_sum = {}
    for probability in PERTURBATION_PROBABILITIES:
        generators[probability] = np.random.default_rng(PERTURBATION_SEED)
        confusion[probability] = np.zeros((3, 3), dtype=np.int64)
        loss_sum[probability] = 0.0

    started = perf_counter()
    for day_index, date in enumerate(test_dates):
        index = all_dates.index(date)
        history = all_dates[index - HISTORY_DAYS:index]
        for history_date in history:
            if history_date not in statistics:
                statistics[history_date] = compute_daily_statistics(
                    processed_dir / f"{history_date}.npz",
                )
        mean, std = compute_history_statistics([statistics[d] for d in history])

        with np.load(processed_dir / f"{date}.npz", allow_pickle=False) as arrays:
            raw_features = arrays["features"]
        with np.load(label_dir / f"{date}.npz", allow_pickle=False) as arrays:
            labels = arrays["labels"]
        with np.load(feature_dir / f"{date}.npz", allow_pickle=False) as arrays:
            clean_features = arrays["features"]
        np.testing.assert_array_equal(
            normalize_features(raw_features, mean, std).astype(np.float32),
            clean_features,
        )
        del clean_features

        for probability in PERTURBATION_PROBABILITIES:
            features = prepare_perturbed_day(
                raw_features, probability, generators[probability], mean, std,
            )
            dataset = DailyWindowDataset(
                [features], [labels],
                window_length=config["window_length"], horizon=config["horizon"],
            )
            loader = torch.utils.data.DataLoader(
                dataset, batch_size=config["batch_size"],
                shuffle=False, num_workers=0, drop_last=False,
            )
            daily_metrics = run_epoch(model, loader, criterion, DEVICE, log_every=0)
            confusion[probability] += daily_metrics["confusion"]
            loss_sum[probability] += daily_metrics["loss"] * len(dataset)
            del features, dataset, loader

        statistics[date] = (
            len(raw_features),
            raw_features.mean(axis=0, dtype=np.float64),
            raw_features.std(axis=0, ddof=0, dtype=np.float64),
        )
        del raw_features, labels
        month_finished = day_index + 1 == len(test_dates) or test_dates[day_index + 1][:6] != date[:6]
        if month_finished:
            print(f"{stock}: finished {date[:6]}, {day_index + 1}/{len(test_dates)} days; {(perf_counter() - started) / 60:.1f} min", flush=True)

    stock_metrics = {}
    for probability in PERTURBATION_PROBABILITIES:
        metrics = classification_metrics(confusion[probability])
        np.testing.assert_array_equal(metrics["support"], baseline_metrics[stock]["support"])
        metrics["loss"] = loss_sum[probability] / int(metrics["support"].sum())
        stock_metrics[probability] = metrics
    return stock_metrics


In [12]:
results = {}
for stock in STOCKS:
    print(f"Starting {stock}: five probabilities, 62 test days.", flush=True)
    results[stock] = {0.0: baseline_metrics[stock]}
    results[stock].update(evaluate_stock(stock))
    print(f"Finished {stock}.", flush=True)


Starting KGHM: five probabilities, 62 test days.
KGHM: finished 201710, 22/62 days; 20.0 min
KGHM: finished 201711, 43/62 days; 36.1 min
KGHM: finished 201712, 62/62 days; 50.2 min
Finished KGHM.
Starting PEKAO: five probabilities, 62 test days.
PEKAO: finished 201710, 22/62 days; 9.9 min
PEKAO: finished 201711, 43/62 days; 19.8 min
PEKAO: finished 201712, 62/62 days; 28.7 min
Finished PEKAO.
Starting PKNORLEN: five probabilities, 62 test days.
PKNORLEN: finished 201710, 22/62 days; 25.0 min
PKNORLEN: finished 201711, 43/62 days; 45.0 min
PKNORLEN: finished 201712, 62/62 days; 65.2 min
Finished PKNORLEN.
Starting PKOBP: five probabilities, 62 test days.
PKOBP: finished 201710, 22/62 days; 17.6 min
PKOBP: finished 201711, 43/62 days; 34.2 min
PKOBP: finished 201712, 62/62 days; 48.6 min
Finished PKOBP.
Starting PZU: five probabilities, 62 test days.
PZU: finished 201710, 22/62 days; 17.9 min
PZU: finished 201711, 43/62 days; 34.4 min
PZU: finished 201712, 62/62 days; 51.7 min
Finished P

## 5. Compare overall scores

Each row covers one stock and P across the complete test period. Precision, recall, and F1 are **weighted by true class support**; weighted recall equals accuracy here. Macro F1 weights the three classes equally. Scores are percentages, changes from P=0% are percentage points (pp), and loss is mean cross-entropy.

P=0% reuses experiment 3's CUDA scores; new rows use the device in Section 1. This comparison uses one trained model and one perturbation seed per stock; small device-related numerical differences are possible.


In [13]:
overall_rows = []
class_rows = []
for stock in STOCKS:
    config = checkpoints[stock]["config"]
    baseline = results[stock][0.0]
    for probability, metrics in results[stock].items():
        support = metrics["support"]
        overall_rows.append({
            "Stock": stock,
            "P (%)": int(round(100 * probability)),
            "Samples": int(support.sum()),
            "Accuracy (%)": 100 * float(metrics["accuracy"]),
            "Precision weighted (%)": 100 * float(np.dot(metrics["precision"], support) / support.sum()),
            "Recall weighted (%)": 100 * float(np.dot(metrics["recall"], support) / support.sum()),
            "F1 weighted (%)": 100 * float(metrics["weighted_f1"]),
            "F1 macro (%)": 100 * float(np.mean(metrics["f1"])),
            "Loss": float(metrics["loss"]),
            "Accuracy change (pp)": 100 * float(metrics["accuracy"] - baseline["accuracy"]),
            "F1 change (pp)": 100 * float(metrics["weighted_f1"] - baseline["weighted_f1"]),
        })
        for class_index, class_name in enumerate(config["class_names"]):
            class_rows.append({
                "Stock": stock,
                "P (%)": int(round(100 * probability)),
                "Class": class_name,
                "Support": int(support[class_index]),
                "Precision (%)": 100 * float(metrics["precision"][class_index]),
                "Recall (%)": 100 * float(metrics["recall"][class_index]),
                "F1 (%)": 100 * float(metrics["f1"][class_index]),
            })

overall_table = pd.DataFrame(overall_rows)
class_table = pd.DataFrame(class_rows)
score_format = {column: "{:.4f}" for column in overall_table if column not in ("Stock", "P (%)", "Samples")}
display(overall_table.style.hide(axis="index").format(score_format))


Stock,P (%),Samples,Accuracy (%),Precision weighted (%),Recall weighted (%),F1 weighted (%),F1 macro (%),Loss,Accuracy change (pp),F1 change (pp)
KGHM,0,3765697,66.8938,68.5730,66.8938,67.2066,66.8805,0.7427,0.0000,0.0000
KGHM,1,3765697,50.9545,50.6698,50.9545,48.8717,47.7049,1.0833,-15.9393,-18.3349
KGHM,5,3765697,41.1994,41.9877,41.1994,36.0890,34.3427,1.3388,-25.6944,-31.1177
KGHM,10,3765697,39.5586,40.9400,39.5586,33.7423,31.9881,1.4093,-27.3352,-33.4643
KGHM,25,3765697,38.9316,41.6869,38.9316,31.5336,29.8577,1.4673,-27.9622,-35.6730
KGHM,50,3765697,41.4664,43.2634,41.4664,35.9123,34.3051,1.3053,-25.4274,-31.2943
PEKAO,0,1991235,64.7312,65.7613,64.7312,65.0079,64.7833,0.7873,0.0000,0.0000
PEKAO,1,1991235,48.7033,50.5954,48.7033,43.2924,42.2940,1.0803,-16.0278,-21.7156
PEKAO,5,1991235,40.8524,44.5706,40.8524,33.6951,32.5638,1.2174,-23.8788,-31.3128
PEKAO,10,1991235,39.6620,42.1571,39.6620,32.5941,31.4834,1.2389,-25.0692,-32.4138


## 6. Inspect each class

Display precision, recall, F1, and support separately for **down, stationary, and up**, so class-specific effects are visible. Support stays fixed across P because labels and eligible windows are unchanged.


In [14]:
for stock in STOCKS:
    print(stock)
    stock_table = class_table.loc[class_table["Stock"] == stock].drop(columns="Stock")
    display(stock_table.style.hide(axis="index").format({
        "Precision (%)": "{:.4f}", "Recall (%)": "{:.4f}", "F1 (%)": "{:.4f}",
    }))


KGHM


P (%),Class,Support,Precision (%),Recall (%),F1 (%)
0,down,1351666,74.0300,65.8172,69.6825
0,stationary,1086291,54.8704,71.4482,62.0715
0,up,1327740,74.2284,64.2636,68.8875
1,down,1351666,50.7596,64.5696,56.8378
1,stationary,1086291,49.3344,21.9820,30.4128
1,up,1327740,51.6710,60.7980,55.8641
5,down,1351666,42.4616,50.8473,46.2776
5,stationary,1086291,43.7651,4.5788,8.2902
5,up,1327740,40.0512,61.3387,48.4602
10,down,1351666,41.7832,41.3889,41.5851


PEKAO


P (%),Class,Support,Precision (%),Recall (%),F1 (%)
0,down,697130,71.7091,65.0760,68.2317
0,stationary,616074,53.8833,64.2869,58.6272
0,up,678031,70.4385,64.7804,67.4910
1,down,697130,49.2755,67.7595,57.0579
1,stationary,616074,55.5750,8.5103,14.7604
1,up,678031,47.4278,65.6306,55.0639
5,down,697130,41.5034,60.6263,49.2745
5,stationary,616074,53.0001,0.6868,1.3560
5,up,678031,40.0649,57.0169,47.0608
10,down,697130,39.7669,62.2696,48.5369


PKNORLEN


P (%),Class,Support,Precision (%),Recall (%),F1 (%)
0,down,1636304,72.7384,65.8205,69.1067
0,stationary,1039891,51.8650,63.9630,57.2822
0,up,1637508,71.3910,67.6006,69.4441
1,down,1636304,60.6369,64.7837,62.6418
1,stationary,1039891,45.0963,38.6414,41.6200
1,up,1637508,61.1053,62.4838,61.7869
5,down,1636304,54.6478,60.2534,57.3139
5,stationary,1039891,39.2011,30.3687,34.2242
5,up,1637508,53.5265,55.6987,54.5910
10,down,1636304,53.6290,58.4497,55.9357


PKOBP


P (%),Class,Support,Precision (%),Recall (%),F1 (%)
0,down,1269370,73.1384,64.8520,68.7464
0,stationary,1035491,52.5224,72.5619,60.9369
0,up,1301745,77.3498,62.4195,69.0872
1,down,1269370,69.5420,65.8513,67.6464
1,stationary,1035491,52.5127,65.3150,58.2183
1,up,1301745,73.8290,63.3321,68.1789
5,down,1269370,59.7370,69.4568,64.2313
5,stationary,1035491,52.7384,39.1138,44.9156
5,up,1301745,63.6954,66.6788,65.1530
10,down,1269370,54.6455,70.8082,61.6857


PZU


P (%),Class,Support,Precision (%),Recall (%),F1 (%)
0,down,1357339,72.9514,62.2438,67.1736
0,stationary,1210997,51.9147,69.1478,59.3047
0,up,1340729,72.6208,61.6379,66.6801
1,down,1357339,69.4315,62.5105,65.7895
1,stationary,1210997,51.9212,63.7756,57.2411
1,up,1340729,69.1356,61.8551,65.2930
5,down,1357339,59.8528,63.6127,61.6755
5,stationary,1210997,51.8746,46.5473,49.0668
5,up,1340729,60.3104,62.0693,61.1772
10,down,1357339,53.9709,64.3396,58.7009


## 7. Save the results

Save the two tables as `metrics.csv` and `class_metrics.csv`. Save confusion matrices, losses, and evaluation settings in `test_metrics.npz`.

The NumPy stock and probability axes follow `STOCKS` and `[0, 0.01, 0.05, 0.10, 0.25, 0.50]`. Confusion-matrix rows are true classes and columns are predicted classes, ordered down, stationary, up.


In [15]:
overall_table.to_csv(EXPERIMENT_DIR / "metrics.csv", index=False)
class_table.to_csv(EXPERIMENT_DIR / "class_metrics.csv", index=False)
all_probabilities = (0.0,) + PERTURBATION_PROBABILITIES
confusion_matrices = np.stack([
    np.stack([results[stock][p]["confusion"] for p in all_probabilities])
    for stock in STOCKS
])
losses = np.asarray([
    [results[stock][p]["loss"] for p in all_probabilities]
    for stock in STOCKS
])
np.savez_compressed(
    EXPERIMENT_DIR / "test_metrics.npz",
    stocks=np.asarray(STOCKS), probabilities=np.asarray(all_probabilities),
    confusion=confusion_matrices, loss=losses,
    class_names=np.asarray(checkpoints[STOCKS[0]]["config"]["class_names"]),
    perturbation_seed=PERTURBATION_SEED, tick_size=TICK_SIZE,
    history_days=HISTORY_DAYS, window_length=50, horizon=20, batch_size=64,
    label_alpha=np.asarray([checkpoints[s]["config"]["label_alpha"] for s in STOCKS]),
    selected_epoch=np.asarray([checkpoints[s]["epoch"] for s in STOCKS]),
    test_dates=np.asarray([checkpoints[s]["config"]["data_dates"]["test"] for s in STOCKS]),
    evaluation_device=str(DEVICE), baseline_source="experiment_03 saved CUDA evaluation",
    python=sys.version.split()[0], numpy=np.__version__, pytorch=torch.__version__,
)
print("Saved metrics.csv, class_metrics.csv, and test_metrics.npz.")


Saved metrics.csv, class_metrics.csv, and test_metrics.npz.
